# MaS-TransUNet — Colab Training Notebook

This notebook trains and evaluates the MaS-TransUNet model for medical image segmentation.
Run each cell in order. Read the markdown instructions before each cell carefully.

---
## 1. Mount Google Drive

This mounts your Google Drive to `/content/drive` so datasets and checkpoints persist across sessions.

**Before proceeding:** Make sure you are logged into the Google account that contains your datasets under `My Drive/datasets/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 2. Install Dependencies

Installs all required packages and verifies CUDA is available.

**Before proceeding:** Check that the output shows a GPU name (e.g. `Tesla T4`). If it says CUDA is not available, go to `Runtime > Change runtime type` and select **GPU**.

In [ ]:
!pip install -q torch torchvision timm==0.9.12 einops albumentations \
    opencv-python-headless scikit-image scikit-learn monai matplotlib \
    seaborn pandas tqdm tensorboard scipy Pillow

import torch
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is NOT available. Training will be very slow.")
    print("Go to Runtime > Change runtime type > GPU")

---
## 3. Clone Repository

Clones the MaS-TransUNet repository from GitHub and enters the project directory.

**Before proceeding:** Replace `YOUR_USERNAME` with your actual GitHub username. If the repo is private, you may need a personal access token. On reconnection, `git pull` ensures you have the latest code.

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/TransUNet.git"
REPO_DIR = "/content/TransUNet"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!git pull
print(f"Working directory: {os.getcwd()}")

---
## 4. Set Colab Mode

Switches all config paths to Google Drive locations and sets the device to CUDA.

**Before proceeding:** Verify each dataset path prints **Found**. If any path prints **Missing**, upload the dataset to the corresponding Drive folder before training on that dataset.

In [ ]:
import os
from config import CFG

# Switch to Colab paths
CFG.is_colab = True
CFG.__post_init__()  # Re-derive paths with Colab flag
CFG.device = "cuda"

print(f"Base data dir:   {CFG.base_data_dir}")
print(f"Checkpoint dir:  {CFG.checkpoint_dir}")
print(f"Device:          {CFG.device}")
print()

for name, paths in CFG.dataset_paths.items():
    img_dir = paths['train_images']
    status = "Found" if os.path.isdir(img_dir) else "Missing"
    print(f"  {name:12s}  {status:7s}  {img_dir}")

---
## 5. Create Directories

Creates the checkpoint and log directories on Drive if they do not already exist.

**Before proceeding:** Confirm both directories show `exists: True`.

In [ ]:
import os
from config import CFG

os.makedirs(CFG.checkpoint_dir, exist_ok=True)
os.makedirs(CFG.log_dir, exist_ok=True)

print(f"Checkpoint dir: {CFG.checkpoint_dir}  (exists: {os.path.isdir(CFG.checkpoint_dir)})")
print(f"Log dir:        {CFG.log_dir}  (exists: {os.path.isdir(CFG.log_dir)})")

---
## 6. Preprocess Datasets

Generates Canny edge maps for all training and test masks. This only runs if edge map directories are empty or missing, so it is safe to re-run after a Colab reconnection without redoing work.

**Before proceeding:** Check the summary. Each dataset with data present should show a non-zero edge map count. Datasets marked as not found are skipped.

In [ ]:
import os
from config import CFG

needs_preprocessing = False
for name, paths in CFG.dataset_paths.items():
    edge_dir = paths['train_edges']
    masks_dir = paths['train_masks']
    if os.path.isdir(masks_dir):
        if not os.path.isdir(edge_dir) or len(os.listdir(edge_dir)) == 0:
            needs_preprocessing = True
            print(f"  {name}: edge maps missing -- will preprocess")
        else:
            print(f"  {name}: edge maps already exist ({len(os.listdir(edge_dir))} files) -- skipping")
    else:
        print(f"  {name}: dataset not found -- skipping")

if needs_preprocessing:
    print("\nRunning preprocessing...")
    !python preprocess.py all
else:
    print("\nAll edge maps already exist. No preprocessing needed.")

---
## 7. Smoke Test

Instantiates the MaS-TransUNet model, runs a single forward pass with dummy data, and prints output shapes and parameter count. This catches import errors, shape mismatches, or GPU memory issues before committing to a full training run.

**Before proceeding:** Verify all four output shapes are printed, the parameter count is ~775M, and there are no errors.

In [ ]:
import torch
from config import CFG
from models import build_model

model = build_model(CFG)

dummy_image = torch.randn(1, 3, 224, 224, device=CFG.device)
dummy_mask  = torch.zeros(1, 1, 224, 224, device=CFG.device)

with torch.no_grad():
    outputs = model(dummy_image, dummy_mask)

print("Output shapes:")
for key, val in outputs.items():
    print(f"  {key:10s}: {tuple(val.shape)}")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params / 1e6:.2f}M")
print("Smoke test PASSED")

# Free memory
del model, dummy_image, dummy_mask, outputs
torch.cuda.empty_cache()

---
## 8. Select Dataset and Train

Trains MaS-TransUNet on the selected dataset. Change `DATASET_NAME` at the top of the cell to switch datasets — no other changes are needed.

Training automatically resumes from the latest checkpoint if a Colab session disconnects. Checkpoints and previous-epoch masks are saved to Google Drive every 5 epochs.

**Before proceeding:** Make sure the dataset you selected showed **Found** in Cell 4. Monitor the progress bar for `loss` decreasing and `dice` increasing.

In [ ]:
DATASET_NAME = "tcga_lgg"  # <-- Change this to train a different dataset

# Options: "tcga_lgg", "dsb2018", "kvasir_seg", "isic2018", "covid_ct"

!python train.py {DATASET_NAME}

---
## 9. Evaluate

Runs evaluation on the test set using the best checkpoint and iterative FAM refinement (up to 10 iterations per sample). Prints a metrics table matching Tables II–VI in the paper, saves per-sample results to CSV, and writes qualitative side-by-side PNGs for the first 5 test images.

**Before proceeding:** Make sure training (Cell 8) has completed at least a few epochs and a `_best.pth` checkpoint exists.

In [ ]:
# Uses the same DATASET_NAME from Cell 8
!python evaluate.py {DATASET_NAME}

---
## 10. Training Order Note

The following cell explains the recommended order for training across all five datasets.

### Recommended Training Order

Train datasets in this order for fastest iteration and debugging:

| Order | Dataset      | Size     | Notes |
|:-----:|:-------------|:---------|:------|
| 1     | `tcga_lgg`   | Smallest | Start here to verify the full pipeline end-to-end with minimal time. Good for catching bugs early. |
| 2     | `dsb2018`    | Small    | Data Science Bowl 2018 nuclei segmentation. Quick to train, validates generalisation to cell-level tasks. |
| 3     | `kvasir_seg` | Medium   | Gastrointestinal polyp segmentation with more complex boundaries. Tests the EAM and edge loss. |
| 4     | `isic2018`   | Large    | Skin lesion segmentation with high variability. Longer training; monitor for overfitting. |
| 5     | `covid_ct`   | Largest  | COVID-19 CT scan segmentation. Most computationally expensive. Train last once everything else works. |

**To switch datasets:** Go back to **Cell 8**, change `DATASET_NAME`, and re-run Cells 8 and 9.

**Checkpointing:** All checkpoints are saved to Google Drive, so training resumes automatically after a Colab disconnection. Simply re-run from Cell 1.